## 分词

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
     "C:\llama2\llama-2-7b-hf",
     offload_folder="offload",
)
tokenizer

LlamaTokenizerFast(name_or_path='C:\llama2\llama-2-7b-hf', vocab_size=32000, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
     "C:\llama2\chinese-alpaca-2-7b",
     offload_folder="offload",
)

In [8]:
sentence = "快速的棕色狐狸跳过了懒洋洋的狗。"
tokenized_sentence = tokenizer.tokenize(sentence)
print(tokenized_sentence)

['▁', '快速', '的', '棕色', '狐狸', '跳', '过了', '懒', '洋洋', '的', '狗', '。']


In [9]:
sentence_encoded = tokenizer(sentence)
print(sentence_encoded)

{'input_ids': [1, 29871, 33026, 30210, 46259, 47244, 32791, 33952, 35929, 49841, 30210, 32499, 30267], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [10]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
     "C:\llama2\chinese-alpaca-2-7b",
     offload_folder="offload",
)

tokenizer.pad_token = tokenizer.eos_token

sentence = ["快速的棕色狐狸跳过了懒洋洋的狗。","我是一个学生。"]

tokenized_sentence = tokenizer(sentence, padding = "max_length", max_length = 15)
tokenized_sentence

{'input_ids': [[2, 2, 1, 29871, 33026, 30210, 46259, 47244, 32791, 33952, 35929, 49841, 30210, 32499, 30267], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 29871, 39511, 32176, 30267]], 'attention_mask': [[0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1]]}

## 模型训练（需GPU）

In [ ]:
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
)

model_path = "/chinese-llama-2-7b"

max_length = 128

# Model loading params
#load_in_4bit = True

# LoRA Params
lora_alpha = 16             # How much to weigh LoRA params over pretrained params
lora_dropout = 0.1          # Dropout for LoRA weights to avoid overfitting
lora_r = 16                 # Bottleneck size between A and B matrix for LoRA params
lora_bias = "all"           # "all" or "none" for LoRA bias
#model_type = "llama"     # falcon or llama
lora_target_modules = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj"
]

# Trainer params
output_dir = "outputs"                              # Directory to save the model
optim_type = "adamw_8bit"                           # Optimizer type to train with 
learning_rate = 0.0005                              # Model learning rate
weight_decay = 0.002                                # Model weight decay
per_device_train_batch_size = 1                     # Train batch size on each GPU
per_device_eval_batch_size = 1                      # Eval batch size on each GPU
gradient_accumulation_steps = 16                    # Number of steps before updating model
warmup_steps = 5                                    # Number of warmup steps for learning rate
save_steps = 10                                    # Number of steps before saving model
logging_steps = 10                                 # Number of steps before logging

# Load in the model as a 4-bit or 8-bit model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    #offload_folder="offload", 
    device_map="auto", 
    quantization_config=bnb_config
)

# Load in the tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    #offload_folder="offload",
)
tokenizer.pad_token = tokenizer.eos_token

# Load in the dataset and map using the tokenizer
dataset = load_dataset("squad")

def map_function(example):
    # Get the question and model output
    question = f"#### Human: {example['question'].strip()}"
    output = f"#### Assistant: {example['answers']['text'][0].strip()}"

'''----------------------------------------------------------------
question = "#### Human: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France? "
output = "#### Assistant: Saint Bernadette Soubirous"
----------------------------------------------------------------'''

    # Encode the question and output
    question_encoded = tokenizer(question)
    output_encoded = tokenizer(output, max_length=max_length-len(question_encoded["input_ids"]), truncation=True, padding="max_length")

'''----------------------------------------------------------------
question和output的input_ids长度累加不能超过max_length。由于分词器中定义padding_side='left'，
因此长度不够就在output_encoded的字符串左侧进行填空。
question_encoded:
['▁####', '▁Human', ':', '▁To', '▁whom', '▁did', '▁the', '▁Virgin', 
'▁Mary', '▁alleg', 'edly', '▁appear', '▁in', '▁', '1', '8', '5', '8', 
'▁in', '▁L', 'our', 'des', '▁France', '?', '▁']
{'input_ids': [1, 3191, 12968, 29901, 1763, 6029, 1258, 278, 9167, 6182, 16831, 
23244, 2615, 297, 29871, 29896, 29947, 29945, 29947, 297, 365, 473, 2783, 3444, 
29973, 29871], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 
1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
output_encoded：
['▁####', '▁Ass', 'istant', ':', '▁Saint', '▁Bern', 'ad', 'ette', '▁Sou', 
'bir', 'ous']
{'input_ids': [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 
2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 
2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 
2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 3191, 4007, 22137, 29901, 4107, 6209, 328, 
2353, 9194, 20397, 681], 'attention_mask': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 
0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 
0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 
0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 
1, 1, 1, 1, 1, 1, 1]}
----------------------------------------------------------------'''

    # Combine the input ids
    input_ids = question_encoded["input_ids"] + output_encoded["input_ids"]

'''----------------------------------------------------------------
[1, 3191, 12968, 29901, 1763, 6029, 1258, 278, 9167, 6182, 16831, 23244, 2615, 297, 
29871, 29896, 29947, 29945, 29947, 297, 365, 473, 2783, 3444, 29973, 29871, 2, 2, 2, 
2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 
2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 
2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 
2, 1, 3191, 4007, 22137, 29901, 4107, 6209, 328, 2353, 9194, 20397, 681]
----------------------------------------------------------------'''

    # The labels are the input ids, but we want to mask the loss for the context and padding
    labels = [-100]*len(question_encoded["input_ids"]) + [output_encoded["input_ids"][i] if output_encoded["attention_mask"][i] == 1 else -100 for i in range(len(output_encoded["attention_mask"]))]

'''----------------------------------------------------------------
将output_encoded中，凡是注意力掩码attention_mask不为1（即为填空标记）input_ids均设为-100。
至于选-100这个值的原因，是因为PyTorch的交叉熵损失函数会忽略-100标签。
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 
-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 
-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 
-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 
-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 
-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 
-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 
-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 
-100, -100, 1, 3191, 4007, 22137, 29901, 4107, 6209, 328, 2353, 9194, 20397, 681]
----------------------------------------------------------------'''

    # Combine the attention masks. Attention masks are 0
    # where we want to mask and 1 where we want to attend.
    # We want to attend to both context and generated output
    attention_mask = [1]*len(question_encoded["input_ids"]) + output_encoded["attention_mask"]

'''----------------------------------------------------------------
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 
0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 
0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 
0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 
0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
----------------------------------------------------------------'''
    
    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": attention_mask
    }

data_train = dataset["train"].map(map_function)
data_test = dataset["validation"].map(map_function)

'''----------------------------------------------------------------
dataset：
DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

dataset['train']：
Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 87599
})

data_train：
Dataset({
    features: ['id', 'title', 'context', 'question', 'answers', 'input_ids', 'labels', 'attention_mask'],
    num_rows: 87599
})
----------------------------------------------------------------'''

# Adapt the model with LoRA weights
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias=lora_bias,
    task_type="CAUSAL_LM",
    inference_mode=False,
    target_modules=lora_target_modules
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir=output_dir,
    evaluation_strategy="epoch",
    optim=optim_type,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    do_train=True,
    warmup_steps=warmup_steps,
    save_steps=save_steps,
    logging_steps=logging_steps,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=data_train,
    eval_dataset=data_test,
    tokenizer=tokenizer,
)

# Train the model
trainer.train()

## 合并权重文件

In [ ]:
import peft
import torch
from peft import PeftConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, HfArgumentParser
import shutil

lora_path = "outputs/checkpoint-100" # Path to the LoRA weights
output_path = "outputs/merged_model" # Path to output the merged weights

peft_model_id = lora_path
peft_config = PeftConfig.from_pretrained(peft_model_id)

'''--------------------------------------------------------------------
从增量模型中读取基础模型的数据，如文件位置base_model_name_or_path。
LoraConfig(peft_type='LORA', auto_mapping=None, base_model_name_or_path
='/mnt/chinese-llama-2-7b', revision=None, task_type='CAUSAL_LM', 
inference_mode=True, r=16, target_modules=['q_proj', 'k_proj', 'v_proj', 
'o_proj', 'gate_proj', 'up_proj', 'down_proj'], lora_alpha=16, 
lora_dropout=0.1, fan_in_fan_out=False, bias='all', modules_to_save=None, 
init_lora_weights=True, layers_to_transform=None, layers_pattern=None)
---------------------------------------------------------------------'''

# Save the model
model = AutoModelForCausalLM.from_pretrained(
    peft_config.base_model_name_or_path,
    return_dict=True,
    torch_dtype=torch.float16,
    trust_remote_code=True,
    device_map="cpu",
)
tokenizer = AutoTokenizer.from_pretrained(peft_config.base_model_name_or_path)

'''--------------------------------------------------------------------
和模型训练时一样，加载基础模型和分词器，这里就是加载chinese-llama-2-7b模型。
--------------------------------------------------------------------'''

shutil.copytree(peft_config.base_model_name_or_path, output_path, dirs_exist_ok=True, 
                ignore=shutil.ignore_patterns('*.pt', "*.pth", "*.bin"))

'''--------------------------------------------------------------------
hutil.copytree用于递归地复制整个目录树。它可以将源目录及其所有子目录和文件复制到目标位置。
dirs_exist_ok参数：默认为False。当该参数设置为True时，如果目标目录已经存在，它将不会引发
FileExistsError异常，而是继续执行复制操作并覆盖已存在的目录。如果设置为False，则如果目标
目录已存在，则会引发FileExistsError异常。
ignore参数：它接受一个可调用对象，用于指定在复制过程中应该被忽略的文件和文件夹。可调用对
象被传递一个参数，即当前正在处理的目录的文件和文件夹列表。您可以根据自己的需求定制这个可
调用对象，以决定应该忽略哪些文件或文件夹。
shutil.ignore_patterns功能：这是一个用于创建忽略模式的辅助函数。这里要求在拷贝时忽略后缀
为pt、pth、bin的文件。支持使用*和?通配符进行匹配。
--------------------------------------------------------------------'''

# Load the Lora model
model = PeftModel.from_pretrained(model, peft_model_id)
model.eval()

'''--------------------------------------------------------------------
利用PeftModel类，把基础模型和增量模型合二为一。
--------------------------------------------------------------------'''

key_list = [key for key, _ in model.base_model.model.named_modules() if "lora" not in key]
for key in key_list:
    try:
        sub_mod = model.get_submodule(key)
        parent = model.get_submodule(".".join(key.split(".")[:-1]))

'''--------------------------------------------------------------------
通过去除最后一项，获取上一级模块的名称
key = "model.embed_tokens.modules_to_save.default"
print(key.split("."))
print(".".join(key.split(".")))
print(".".join(key.split(".")[:-1]))
输出：
['model', 'embed_tokens', 'modules_to_save', 'default']
model.embed_tokens.modules_to_save.default
model.embed_tokens.modules_to_save
--------------------------------------------------------------------'''

    except AttributeError:
        continue
    target_name = key.split(".")[-1]
    
'''--------------------------------------------------------------------
# 获取键的最后一部分（即子模块的名称）
--------------------------------------------------------------------'''

    if isinstance(sub_mod, peft.tuners.lora.Linear):
        sub_mod.merge()
        bias = sub_mod.bias is not None
        new_module = torch.nn.Linear(sub_mod.in_features, sub_mod.out_features, bias=bias)
        new_module.weight.data = sub_mod.weight
        if bias:
            new_module.bias.data = sub_mod.bias
        model.base_model._replace_module(parent, target_name, new_module, sub_mod)
        
'''--------------------------------------------------------------------
如果sub_mod是peft.tuners.lora.Linear类型的实例
创建一个新的线性模块，具有与子模块相同的输入特征和输出特征，同时考虑是否有偏置
将新模块的权重参数设置为子模块的权重参数
如果存在偏置，将新模块的偏置参数设置为子模块的偏置参数
用新模块替换原有模型中指定子模块的位置       
--------------------------------------------------------------------'''

model.save_pretrained(output_path)

## 推理

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    AutoTokenizer,
)

device = "auto"
model_path = "outputs/merged_model"             # Path to the combined weights

# Prompt should be in this style due to how the data was created
prompt = "#### Human: What is the capital of Australia?#### Assistant:"

bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype="float16",
        bnb_4bit_use_double_quant=True,
    )

model = AutoModelForCausalLM.from_pretrained(
    model_path, 
    trust_remote_code=True, 
    device_map=device, 
    # load_in_8bit=True,
    quantization_config=bnb_config
)
tokenizer = AutoTokenizer.from_pretrained(model_path)

inputs = tokenizer(prompt, return_tensors="pt")
if device != "cpu":
    inputs = inputs.to('cuda')
del inputs['token_type_ids']

output = model.generate(**inputs, do_sample=True, top_p=0.95, top_k=60, max_new_tokens=100)

'''--------------------------------------------------------------------
这个推理函数通过调整不同的参数来生成文本。下面是对这些参数的定义：
1. **inputs：这是一个字典类型的参数，其中包含了模型输入的各种信息，如输入的文本、编码形
式等。这些信息通常包括"input_ids"、"attention_mask"和"token_type_ids"等。
2. do_sample：这是一个布尔型参数，默认为True。当设置为True时，模型将使用采样的方式生成
文本，即根据概率分布随机选择下一个词。这有助于生成更加多样化和创造性的文本。
3. top_p：这是一个浮点型参数，取值范围为0到1，默认为0.95。它表示生成文本时，模型将在累
积概率分布超过该阈值时停止生成新的词。较小的值会增加生成文本的多样性，而较大的值会使生
成文本更加准确和确定性。
4. top_k：这是一个整型参数，默认为60。它定义了在生成文本时，模型只考虑概率最高的前k个词。
较小的值会增加生成文本的多样性，而较大的值会使生成文本更加准确和确定性。
5. max_new_tokens：这是一个整型参数，默认为100。它限制了生成文本的最大长度，以控制生成
文本的输出长度。
--------------------------------------------------------------------'''

output = tokenizer.decode(output[0], skip_special_tokens=True)

'''--------------------------------------------------------------------
<s> #### Assistant: Saint Bernadette Soubirous
设置了参数skip_special_tokens=True，可以去掉<s>
--------------------------------------------------------------------'''

print(output.split("#### Assistant: ")[1])

'''--------------------------------------------------------------------
去掉输出文本中的”#### Assistant:”，显示不带任何附加格式的文本。
--------------------------------------------------------------------'''

## 加载多个LoRA模型，并随时切换（需GPU）

In [ ]:
from peft import PeftModel
from transformers import LlamaTokenizer, LlamaForCausalLM, GenerationConfig

model_name = "c:/llama2/llama-2-7b-hf"

tokenizer = LlamaTokenizer.from_pretrained(model_name)
model = LlamaForCausalLM.from_pretrained(
    model_name,
    load_in_8bit=True,
    device_map="auto",
    use_auth_token=True
)

model = PeftModel.from_pretrained(model, "tloen/alpaca-lora-7b", adapter_name="eng_alpaca")
model.load_adapter("22h/cabrita-lora-v0-1", adapter_name="portuguese_alpaca")

model.set_adapter("eng_alpaca")
instruction = "Tell me about alpacas."
print(evaluate(instruction))

"""输出
The alpaca (Vicugna pacos) is a domesticated species of South American camelid. It resembles a small llama in appearance, but unlike the llama, it is not used as a beast of burden. It is kept primarily for its fiber, which can be spun into yarn. Alpaca fiber is warmer, lighter, and softer than sheep's wool, and is highly valued in the textile industry. The fiber comes in a variety of natural colors, including white, beige, cream, and fawn. It can also be dyed in a wide range of colors.
Alpaca herds can be found in the highlands of Peru, Bolivia, Chile, Ecuador, and Colombia. They are also raised in the United States, Canada, Australia, New Zealand, and Europe. The animals graze on grasses, herbs, and shrubs, and can survive in temperatures as low as -30°F (-34°C). They are social animals, living in herds of up to 20 individuals.
The fiber of the alpaka is used to make clothing
"""

model.set_adapter("portuguese_alpaca")
instruction = "Invente uma desculpa criativa pra dizer que não preciso ir à festa."
print(evaluate(instruction))
"""输出
"Eu preciso ficar em casa para cuidar de meu gato."
"""

with model.disable_adapter():
    instruction = "Invente uma desculpa criativa pra dizer que não preciso ir à festa."
    print(evaluate(instruction))
"""输出
I'm sorry, but I can't go to the party. I'm sick. I have a cold. I don't feel well. I need to stay at home and rest.
I have a lot of homework to do. My dog ate my homework. My homework is too hard. I didn't have time to do it. It's too late. I forgot about it.
My parents won't let me go. My parents are out of town. They're on vacation. They have to work. They are sick. They need to take care of my brother.
They're not home. They went to the grocery store. They took the car to the mechanic. They had to go to a meeting. They were in a hurry. They forgot about me.
Their car broke down. Their car ran out of gas. They got a flat tire. They couldn't find a parking space. They didn' t have enough money. They lost their wallet.
It's raining. The roads are icy. There's a blizzard. There are too many cars on the road. There was an accident.
"""